# 🔢 Notebook 4: Embeddings
### How meaning becomes numbers

**WAT Reference:** `workflows/04_embed_chunks.md` | `tools/embed_chunks.py`

---

## The Most Important Concept in RAG

An **embedding** is a list of ~1,536 numbers that captures the *meaning* of a piece of text.

The key insight: **similar meanings produce similar numbers.**

```
"Porter's Five Forces"          → [0.023, -0.441, 0.178, ..., 0.832]  ← 1,536 numbers
"competitive strategy model"    → [0.031, -0.438, 0.181, ..., 0.819]  ← very similar!
"net present value calculation" → [-0.412, 0.233, -0.091, ..., -0.144] ← very different
```

We can measure *how similar* two embeddings are using **cosine similarity** (a math formula). This is what enables semantic search — finding chunks that mean the same thing as your question, even if they use different words.

## What You'll Do

1. See what an embedding looks like
2. Understand cosine similarity intuitively
3. See similar vs dissimilar texts compared numerically
4. **See cost estimate before spending anything**
5. Embed all chunks → save to `.tmp/vector_store.json`

**Time needed:** ~20 minutes | **Cost:** ~$0.001–$0.01 for sample files

In [ ]:
!pip install openai -q
print("✅ OpenAI library ready")

In [ ]:
import os, json, math, time
import openai

# ─────────────────────────────────────────────────────────────────────
# NOTE: This cell mounts Google Drive, which only works in Google Colab.
# If you're running in the browser (via the website), this cell will
# not work — that's expected. To run the full pipeline with your own
# files, open this notebook in Google Colab.
# ─────────────────────────────────────────────────────────────────────
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    IN_COLAB = True
    print("✅ Google Drive mounted successfully!")
except Exception:
    IN_COLAB = False
    print("ℹ️  Google Drive is not available in this environment.")
    print("   To run this notebook with your MBA files, open it in Google Colab.")
    print("   You can still read through the cells to understand how embeddings work.")

PROJECT_ROOT = '/content/drive/MyDrive/MBA_RAG'
INPUT_FILE   = os.path.join(PROJECT_ROOT, '.tmp/chunks.json')
OUTPUT_FILE  = os.path.join(PROJECT_ROOT, '.tmp/vector_store.json')
CHECKPOINT   = os.path.join(PROJECT_ROOT, '.tmp/embed_checkpoint.json')

# ─────────────────────────────────────────────────────────────────────
# Load API key — works in both environments:
#
#   On the website:  key is injected automatically when you click
#                    "Run in browser" and enter it in the panel above.
#
#   In Google Colab: key is loaded from Colab Secrets (🔑 in sidebar).
#                    Add secret name: OPENAI_API_KEY
# ─────────────────────────────────────────────────────────────────────
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')

if not OPENAI_API_KEY and IN_COLAB:
    try:
        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY') or ''
        if OPENAI_API_KEY:
            os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
    except Exception:
        pass

if OPENAI_API_KEY:
    print(f"✅ API key loaded. Starts with: {OPENAI_API_KEY[:8]}...")
else:
    print("⚠️  API key not found.")
    print("   On the website:  click 'Run in browser' and enter your key first.")
    print("   In Google Colab: add it via the 🔑 Secrets panel in the sidebar.")

client = openai.OpenAI(api_key=OPENAI_API_KEY)

# Load chunks from Notebook 3 (only if Drive is available)
if IN_COLAB and os.path.exists(INPUT_FILE):
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        all_chunks = json.load(f)
    print(f"✅ Loaded {len(all_chunks)} chunks from chunks.json")
elif IN_COLAB:
    print(f"⚠️  chunks.json not found at {INPUT_FILE}")
    print("   Run Notebook 3 first to generate it.")
    all_chunks = []
else:
    all_chunks = []

---
## Step 1: What Does an Embedding Look Like?

Let's call the OpenAI API and see what we get back.

In [ ]:
# Embed a single piece of text to see what it looks like
sample_text = "Porter's Five Forces is a framework for analyzing competitive dynamics"

print(f"Text: \"{sample_text}\"")
print()
print("Calling OpenAI API...")

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=sample_text
)

embedding = response.data[0].embedding

print(f"\n✅ Embedding generated!")
print(f"   Number of dimensions: {len(embedding)}")
print(f"   First 10 numbers:     {[round(x, 4) for x in embedding[:10]]}")
print(f"   Last 10 numbers:      {[round(x, 4) for x in embedding[-10:]]}")
print()
print("These 1,536 numbers are meaningless individually.")
print("Their PATTERN and RELATIONSHIP to other embeddings is what matters.")
print("Similar texts will have similar patterns.")

---
## Step 2: Cosine Similarity — Measuring Closeness

Think of each embedding as an **arrow pointing in a direction** in 1,536-dimensional space.

- Similar texts → arrows pointing in the **same direction** → small angle → high similarity
- Different texts → arrows pointing in **different directions** → large angle → low similarity

**Cosine similarity** measures the cosine of that angle:
- Score **1.0** = identical direction = identical meaning
- Score **0.5–0.9** = similar direction = related topics
- Score **0.0** = perpendicular = completely unrelated

The formula: `dot(a, b) / (norm(a) × norm(b))`

We build this from scratch using basic math — no extra libraries needed.

In [ ]:
def cosine_similarity(vec_a, vec_b):
    """
    Compute cosine similarity between two vectors.

    dot_product: sum of element-wise products
    norm: square root of sum of squares (the 'length' of the vector)

    This is pure Python — no numpy needed. Slow for large datasets
    but perfect for understanding the concept.
    """
    dot_product = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)


def get_embedding(text):
    """Get embedding vector for a text string."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


print("✅ Functions defined.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# DEMO: Compare similar vs dissimilar texts
# This shows WHY embeddings enable semantic search
# ─────────────────────────────────────────────────────────────

test_pairs = [
    # (text_a, text_b, expected: similar or different?)
    ("Porter's Five Forces",            "competitive strategy framework",        "SIMILAR"),
    ("Porter's Five Forces",            "five forces competitive analysis",       "SIMILAR"),
    ("Porter's Five Forces",            "net present value calculation",          "DIFFERENT"),
    ("discounted cash flow",            "DCF valuation model",                    "SIMILAR"),
    ("discounted cash flow",            "brand positioning strategy",             "DIFFERENT"),
    ("customer lifetime value",         "CLV marketing metric",                   "SIMILAR"),
]

print("SIMILARITY DEMO")
print("=" * 80)
print(f"{'Text A':<35} {'Text B':<35} {'Score':>6} {'Expected'}")
print("─" * 80)

for text_a, text_b, expected in test_pairs:
    emb_a = get_embedding(text_a)
    emb_b = get_embedding(text_b)
    score = cosine_similarity(emb_a, emb_b)
    print(f"{text_a[:33]:<35} {text_b[:33]:<35} {score:>6.3f} {expected}")

print()
print("💡 Notice:")
print("   - Similar concepts score 0.7-0.95 even with different words")
print("   - Unrelated concepts score 0.3-0.5")
print("   - This is why RAG finds relevant chunks even if exact words differ")

---
## Step 3: Cost Estimate — Before Spending Anything

Embedding calls cost money. This cell calculates the exact cost BEFORE you run anything. You confirm before proceeding.

In [ ]:
# ─────────────────────────────────────────────────────────────
# COST ESTIMATOR — runs with no API calls, no charge
# ─────────────────────────────────────────────────────────────

COST_PER_1M_TOKENS = 0.02  # text-embedding-3-small pricing (USD)

total_chunks = len(all_chunks)
total_tokens = sum(c.get('token_count', 500) for c in all_chunks)
estimated_cost = (total_tokens / 1_000_000) * COST_PER_1M_TOKENS

print("EMBEDDING COST ESTIMATE")
print("=" * 50)
print(f"  Chunks to embed:   {total_chunks:,}")
print(f"  Total tokens:      {total_tokens:,}")
print(f"  Model:             text-embedding-3-small")
print(f"  Price:             ${COST_PER_1M_TOKENS} per 1M tokens")
print(f"  ─────────────────────────────────────")
print(f"  Estimated cost:    ${estimated_cost:.5f} USD")
print()

# Scale to show full corpus cost
scale_to_full = 1250 / max(len(set(c['filename'] for c in all_chunks)), 1)
full_corpus_est = estimated_cost * scale_to_full
print(f"  If you have {len(set(c['filename'] for c in all_chunks))} sample files,")
print(f"  full corpus (1,250 files) would cost ~${full_corpus_est:.2f}")
print()
print("Embeddings run ONCE. Every question after costs only ~$0.001.")

---
## Step 4: Embed All Chunks

▶ Run this cell only after confirming the cost above is acceptable.

Progress is saved every 50 chunks — if interrupted, re-run and it resumes automatically.

In [ ]:
BATCH_SIZE = 50  # chunks per API call

# Check for existing checkpoint (resume if interrupted)
embedded_chunks = []
start_index = 0
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT, 'r') as f:
        checkpoint = json.load(f)
    embedded_chunks = checkpoint['embedded_chunks']
    start_index = len(embedded_chunks)
    print(f"Resuming from checkpoint: {start_index} chunks already embedded")

remaining = all_chunks[start_index:]

print(f"Embedding {len(remaining)} chunks in batches of {BATCH_SIZE}...")
print()

for batch_start in range(0, len(remaining), BATCH_SIZE):
    batch = remaining[batch_start : batch_start + BATCH_SIZE]
    texts = [c['text'] for c in batch]
    done  = start_index + batch_start + len(batch)

    try:
        response = client.embeddings.create(model="text-embedding-3-small", input=texts)
        vectors  = [item.embedding for item in response.data]

        for chunk, vector in zip(batch, vectors):
            entry = dict(chunk)
            entry['embedding'] = vector
            embedded_chunks.append(entry)

        print(f"  ✅ {done}/{len(all_chunks)} chunks embedded")

        # Save checkpoint every 50 chunks
        with open(CHECKPOINT, 'w') as f:
            json.dump({'embedded_chunks': embedded_chunks}, f)

        time.sleep(0.1)  # Gentle pacing to avoid rate limits

    except Exception as e:
        print(f"  ❌ Error at batch {batch_start}: {e}")
        print("  Progress saved. Re-run this cell to resume.")
        break

# Save final output
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(embedded_chunks, f, ensure_ascii=False)

# Remove checkpoint once complete
if os.path.exists(CHECKPOINT) and len(embedded_chunks) == len(all_chunks):
    os.remove(CHECKPOINT)

print()
print("=" * 60)
print(f"✅ Done! {len(embedded_chunks)} chunks embedded")
print(f"   Each chunk now has an 'embedding' field with 1,536 numbers")
print(f"   Saved to: {OUTPUT_FILE}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# VERIFY: Look at one entry in vector_store.json
# ─────────────────────────────────────────────────────────────

sample = embedded_chunks[0]

print("STRUCTURE OF ONE ENTRY IN vector_store.json:")
print("─" * 60)
print(f"  chunk_id:    {sample['chunk_id']}")
print(f"  filename:    {sample['filename']}")
print(f"  subject:     {sample['subject']}")
print(f"  token_count: {sample['token_count']}")
print(f"  text:        {sample['text'][:100]}...")
print(f"  embedding:   [{round(sample['embedding'][0],4)}, {round(sample['embedding'][1],4)}, ... {round(sample['embedding'][-1],4)}]")
print(f"               ← {len(sample['embedding'])} numbers total")
print("─" * 60)
print()
print("This file is your 'vector store' — the heart of the RAG system.")
print("Notebook 5 loads this file and searches it using cosine similarity.")

---
## ✅ Notebook 4 Complete!

### What you learned:
1. **Embedding** = 1,536 numbers that encode the MEANING of text
2. **Similar texts produce similar numbers** — that's the magic
3. **Cosine similarity** measures closeness using the dot product formula
4. **Cost estimator** runs first — you always see the price before spending
5. **Checkpointing** means you can interrupt and resume safely

### The data flow so far:
```
data/sample/  →  .tmp/parsed_docs.json  →  .tmp/chunks.json  →  .tmp/vector_store.json
 (raw files)       (extracted text)         (500-token pieces)   (pieces + 1,536 numbers)
```

---
### Next: Notebook 5 — Retrieval

Now we load `vector_store.json` and use cosine similarity to find the most relevant chunks for any question. You'll see similarity scores and understand why certain chunks rank higher than others.

**Update README.md — check off Notebook 4 ✅**